In [ ]:
import os
import numpy as np
import moabb
from moabb.datasets import Cho2017
from moabb.paradigms import LeftRightImagery

# Set MOABB logging to only show critical stuff, otherwise it prints too much
moabb.set_log_level("WARNING")

def fetch_gigascience_data(num_subjects=15, save_dir="processed_global"):
    os.makedirs(save_dir, exist_ok=True)
    print(f"Downloading & Formatting GigaScience (Cho2017) Data...")
    
    # 1. Setup the Paradigm
    # We enforce the 8-32Hz bandpass filter right here.
    paradigm = LeftRightImagery(fmin=8.0, fmax=32.0)
    
    # 2. Load the Dataset
    dataset = Cho2017()
    
    # Select our subjects (1 through 15)
    subject_list = list(range(1, num_subjects + 1))
    
    # 3. Magic Extraction
    # get_data() automatically downloads, filters, and epochs the data!
    # Output X is [Trials, Channels, TimeSteps]
    X, labels, metadata = paradigm.get_data(dataset=dataset, subjects=subject_list)
    
    # 4. Convert String Labels to Integers (Left=0, Right=1)
    # MOABB returns string labels like 'left_hand' and 'right_hand'
    y_int = np.where(labels == 'left_hand', 0, 1)
    
    # 5. Save the Tensors
    np.save(os.path.join(save_dir, "GLOBAL_X.npy"), X)
    np.save(os.path.join(save_dir, "GLOBAL_y.npy"), y_int)

    print(f"GIGASCIENCE DATASET SUCCESSFULLY PREPARED!")
    print(f"Total Subjects: {num_subjects}")
    print(f"Total Trials:   {X.shape[0]}")
    print(f"Tensor Shape:   {X.shape}  Tensor is now of 64 dimensions because 64 electrodes")

# Execute! (This might take 5-10 minutes to download the raw data on the first run)
fetch_gigascience_data(num_subjects=15)

c:\Users\risha\AppData\Local\Programs\Python\Python311\Lib\site-packages\moabb\datasets\download.py:97: RuntimeWarning: Setting non-standard config type: "MNE_DATASETS_GIGADB_PATH"
  set_config(key, get_config("MNE_DATA"))


🚀 Downloading & Formatting GigaScience (Cho2017) Data...


c:\Users\risha\AppData\Local\Programs\Python\Python311\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 's3.ap-northeast-1.wasabisys.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
100%|########################################| 203M/203M [00:00<00:00, 204GB/s]
SHA256 hash of downloaded file: 57f2f10056b3c240adc78324872597d9b06b282df537a7763e98467275efe6db
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.
2026-05-14 01:03:41,584 WARNING MainThread moabb.datasets.gigadb Trials demeaned and stacked with zero buffer to create continuous data -- edge effects present
c:\Users\risha\AppData\Local\Programs\Python\Python311\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is be


🌍 GIGASCIENCE DATASET SUCCESSFULLY PREPARED!
Total Subjects: 15
Total Trials:   3080
Tensor Shape:   (3080, 64, 1537)  <-- Notice it is 64 channels now, not 22!


In [ ]:
# THE OLD PREPROCESSOR FOR THE OLDER DATASET
import os
import numpy as np
import scipy.io as sio
from scipy.signal import butter, filtfilt

#1.The Wide-Band Filter a normall 8-30Hz band pass filter
def apply_strict_bandpass(data, lowcut = 8.0, highcut = 30.0, fs = 512.0 , order = 4):
    nyq = 0.5*fs
    b, a = butter(order,[lowcut/nyq, highcut/nyq], btype = 'band')
    return filtfilt(b,a,data,axis = 0)

#the fitlering pipeline
def build_global_dataset(data_dir, save_dir):

    all_X = []
    all_y = []

    os.makedirs(save_dir, exist_ok = True)
    print(f"saving to {save_dir}")

    for sub_id in range(1,10):
        filename = f"A0{sub_id}T.mat"
        filepath = os.path.join(data_dir, filename)

        if not os.path.exists(filepath):
            print(f"Skipping file id {sub_id}")
            continue
        print(f"processing {sub_id}")
        mat_data = sio.loadmat(filepath)
        all_runs = mat_data['data'][0]

        SFREQ = 250
        Action_start, Action_end = int(1.0*SFREQ), int(3.0*SFREQ) 
        Baseline_start, baseline_end = int(-0.2*SFREQ), 0

        sub_trials_added = 0

        for run in all_runs:
            X,y,trial = run['X'].item(), run['y'].item(), run['trial'].item()
            if len(y) == 0: continue #skip non motor functions

            filtered_eeg = apply_strict_bandpass(X[:, :22])

            for i in range(len(trial)):
                start_idx = trial[i][0]
                epoch = filtered_eeg[start_idx + Action_start : start_idx + Action_end, :]
                baseline = filtered_eeg[start_idx + Baseline_start : start_idx + baseline_end, :]

                #stretching wave according to user's resting stage
                b_mean = np.mean(baseline, axis=0)
                b_std = np.std(baseline, axis=0)
                norm_epoch = (epoch - b_mean) / (b_std + 1e-6)

                #voltage gate, to remove artifacts
                gated_epoch = np.clip(norm_epoch, -3.0, 3.0)

                # Global Artifact Reject (If it's still somehow massive, dump it)
                if np.max(np.abs(gated_epoch)) < 10:
                    all_X.append(gated_epoch)
                    all_y.append(y[i][0] - 1)
                    sub_trials_added += 1

        print(f"Added {sub_trials_added} clean trials from Subject {sub_id}.")
    
    # Merge and save all subject specific trials
    # Transpose to PyTorch standard: [Trials, Channels, TimeSteps]
    final_X = np.array(all_X).transpose(0, 2, 1) 
    final_y = np.array(all_y)

    np.save(os.path.join(save_dir, "GLOBAL_X.npy"), final_X)
    np.save(os.path.join(save_dir, "GLOBAL_y.npy"), final_y)

    print("\n Global Dataset Successfully Created")
    print(f"Total Combined Trials: {final_X.shape[0]}")
    print(f"Final Tensor Shape:    {final_X.shape}")

build_global_dataset('TrainingTesting-Data', 'processed_global')

In [38]:
import os
import numpy as np
import moabb
from moabb.datasets import Cho2017
from moabb.paradigms import LeftRightImagery
import warnings

# Silence the urllib3 network logs so your terminal stays clean
warnings.filterwarnings("ignore", module="urllib3")
moabb.set_log_level("ERROR") 

def build_gigascience_global(num_subjects=15, save_dir="processed_global"):
    os.makedirs(save_dir, exist_ok=True)
    print("Downloading & Formatting GigaScience (Cho2017)...")
    
    # 1. MOABB Native Filtering (Replaces our old manual bandpass)
    # This automatically applies an 8-32Hz bandpass at fs=512Hz!
    paradigm = LeftRightImagery(fmin=8.0, fmax=32.0)
    dataset = Cho2017()
    
    # 2. Extract Data
    # X_raw shape: [Trials, 64, 1537]
    X_raw, labels, metadata = paradigm.get_data(dataset=dataset, subjects=list(range(1, num_subjects + 1)))
    
    # Convert labels to Binary (Left=0, Right=1)
    y_int = np.where(labels == 'left_hand', 0, 1)
    print(f"   Raw Data Downloaded: {X_raw.shape}")
    
    # 3. The Reaction-Time Shift
    # Skip the visual reaction (first 1.0s = 512 steps).
    # We slice from 512 to 1537 to get exactly 1025 time steps.
    X_shifted = X_raw[:, :, 256:1281] 
    
    # 4. Trial-by-Trial Z-Score Normalization & Voltage Gating
    all_X_clean = []
    print("   Applying Z-Score Normalization & Strict Voltage Gates...")
    for i in range(X_shifted.shape[0]):
        trial = X_shifted[i] # Shape: [64, 1025]
        
        # Normalize each channel individually to neutralize "loud/quiet" brain variance
        t_mean = np.mean(trial, axis=1, keepdims=True)
        t_std = np.std(trial, axis=1, keepdims=True)
        norm_trial = (trial - t_mean) / (t_std + 1e-6)
        
        # Clip blinks/noise
        gated_trial = np.clip(norm_trial, -3.0, 3.0)
        all_X_clean.append(gated_trial)
        
    final_X = np.array(all_X_clean) 
    
    # 5. Save the Master Tensors
    np.save(os.path.join(save_dir, "GLOBAL_X.npy"), final_X)
    np.save(os.path.join(save_dir, "GLOBAL_y.npy"), y_int)

    print(f"GIGASCIENCE PHASE 1 COMPLETE")
    print(f"Total Trials: {final_X.shape[0]}")
    print(f"Final Shape:  {final_X.shape} (64 Channels, 1025 Timesteps)")

# Execute
build_gigascience_global(num_subjects=1)

2026-05-14 02:11:09,523 WARNING MainThread moabb.datasets.gigadb Trials demeaned and stacked with zero buffer to create continuous data -- edge effects present


   Raw Data Downloaded: (200, 64, 1537)
   Applying Z-Score Normalization & Strict Voltage Gates...
GIGASCIENCE PHASE 1 COMPLETE
Total Trials: 200
Final Shape:  (200, 64, 1025) (64 Channels, 1025 Timesteps)


In [39]:
import os
import numpy as np
import librosa
from scipy.stats import kurtosis

def extract_acoustic_physics(X_data, fs = 512.0):
    num_trials, num_channels, time_steps = X_data.shape
    print(f"Extracting audio and physics features for {num_trials} trial")
    
    global_features = []
    
    for i in range(num_trials):
        trial_features = []
        for ch in range(num_channels):
            signal = X_data[i, ch, :]
        
            #Physics and time domain

            rms = np.sqrt(np.mean(signal**2))
            var = np.var(signal)
            kurt = kurtosis(signal)
            # Zero-Crossing Rate (How many times the wave crosses the 0-voltage line)
            zcr = ((signal[:-1] * signal[1:]) < 0).sum()

            #acoustic and freq domain(librosa will take longer)
            # We use a small hop_length and n_fft to capture the low-frequency EEG rhythms
            S, phase = librosa.magphase(librosa.stft(signal, n_fft=128, hop_length=64))
            
            # Spectral Centroid: The "Center of Mass" of the spectrum
            cent = np.mean(librosa.feature.spectral_centroid(S=S, sr=fs))
            
            # Spectral Rolloff: The frequency below which 85% of the energy lies
            rolloff = np.mean(librosa.feature.spectral_rolloff(S=S, sr=fs, roll_percent=0.85))

            # MFCCs: Mel-Frequency Cepstral Coefficients (Tuned for < 40Hz)
            mfccs = librosa.feature.mfcc(y=signal, sr=fs, n_mfcc=2, n_fft=128, hop_length=64, fmax=40)
            mfcc_1 = np.mean(mfccs[0])
            mfcc_2 = np.mean(mfccs[1])
            
            # Bundle all 8 features for this specific channel
            ch_feats = [rms, var, kurt, zcr, cent, rolloff, mfcc_1, mfcc_2]
            trial_features.extend(ch_feats)
            
        global_features.append(trial_features)

        # Progress Tracker on files
        if (i + 1) % 200 == 0 or (i + 1) == num_trials:
            print(f"   Processed {i+1}/{num_trials} trials")
            
    return np.array(global_features)

#Actual loading of data
filepath_X = "processed_global/GLOBAL_X.npy"
if not os.path.exists(filepath_X):
    print(f"Error: Could not find {filepath_X}. Did Phase 1 finish saving?")
else:
    X_global = np.load(filepath_X)
    
    # Run the heavy extraction
    X_acoustic = extract_acoustic_physics(X_global)
    
    # Save the features
    np.save("processed_global/GLOBAL_Acoustic_Features.npy", X_acoustic)
    
    print(f"Acoustic Features Extracted & Saved!")
    print(f"Final Shape: {X_acoustic.shape}") 


Extracting audio and physics features for 200 trial


c:\Users\risha\AppData\Local\Programs\Python\Python311\Lib\site-packages\librosa\feature\spectral.py:2148: UserWarning: Empty filters detected in mel frequency basis. Some channels will produce empty responses. Try increasing your sampling rate (and fmax) or reducing n_mels.
  mel_basis = filters.mel(sr=sr, n_fft=n_fft, **kwargs)


   Processed 200/200 trials
Acoustic Features Extracted & Saved!
Final Shape: (200, 512)


In [71]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

#The new CAE which acts on difference instead of direct features
class DerivativeCAE(nn.Module):
    def __init__(self, latent_dim=8):
        super(DerivativeCAE, self).__init__()
        
        # ENCODER: Learns the compressed patterns of the derivatives
        # Input shape: [Batch, 22, 499] -> 499 because derivative loses 1 timestep
        self.encoder = nn.Sequential(
            nn.Conv1d(64, 16, kernel_size=5, padding=2), #mixes the 64 electrodes into 16 virtual feature maps by sliding a 5 second time window
            nn.GELU(),
            nn.MaxPool1d(2), # 512 steps #looks at every 2 adjacent time steps and throws away the smaller number, keeping only the max
            nn.Conv1d(16, 8, kernel_size=5, padding=2), #same as 1st convolution but now 16 becomes 8 feature maps Now that Layer 1 found basic "edges," Layer 3 combines those edges to find larger patterns (like "an edge followed by a plateau")
            nn.GELU(),
            nn.MaxPool1d(2), #256 steps #Halves the time dimension one more time
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(8 * 256, latent_dim) #We take the 8 highly-abstracted feature maps (which are now 256 time-steps long) and unroll them into a straight line of 2048 numbers. We then feed those 2048 numbers into a standard Neural Network layer that squashes them into exactly 16 latent features.
        )
        
        # DECODER: Tries to reconstruct the derivative wave
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 8 * 256),
            nn.GELU(),
            nn.Unflatten(1, (8, 256)),
            nn.ConvTranspose1d(8, 16, kernel_size=2, stride=2), # 384 -> 768
            nn.GELU(),
            nn.ConvTranspose1d(16, 64, kernel_size=2, stride=2) # 768 -> 1536 
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

#the derivatives
print("Difference data")
X_global = np.load("processed_global/GLOBAL_X.npy") # [2592, 22, 500]

# Calculate the time derivative (Rate of Change)
# This calculates (t1 - t0), (t2 - t1), etc.
X_diff = np.diff(X_global, axis=2) # Shape becomes [2592, 22, 499]

# Convert to PyTorch Tensors
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tensor_X = torch.from_numpy(X_diff).float()

#because unsupervised learning y dataset not used
dataset = TensorDataset(tensor_X)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Training the differenceCAE
model = DerivativeCAE(latent_dim=16).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss() # Mean Squared Error for reconstruction

# The updated Phase 3 Training Loop with L1 Sparsity
EPOCHS = 30 # Slightly higher to let it adapt to the strict constraints
print("\nTraining Constrained Sparse Derivative AutoEncoder")
model.train()

# The "tuning knob" for how strict we are. 1e-4 is a standard starting point.
l1_weight = 1e-4 

for epoch in range(EPOCHS):
    running_loss = 0.0
    for batch in dataloader:
        inputs = batch[0].to(device)
        optimizer.zero_grad()
        
        # Standard reconstruction loss ONLY
        encoded, decoded = model(inputs)
        loss = criterion(decoded, inputs) 
        
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
    if (epoch+1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}] | Total Loss: {running_loss/len(dataloader):.4f}")

# Extraction and fusion of features
print("\nExtracting AI Features and Fusing")
model.eval()

with torch.no_grad():
    # Pass all data through the encoder at once
    encoded_features, _ = model(tensor_X.to(device))
    X_ai = encoded_features.cpu().numpy() # Shape: [2592, 16]

# Load the Acoustic/Physics features you just generated
X_acoustic = np.load("processed_global/GLOBAL_Acoustic_Features.npy") # Shape: [2592, 176]

# Fuse them together! (Stacking horizontally)
# 176 (Physics) + 16 (AI) = 192 Total Features
X_fused_final = np.hstack((X_acoustic, X_ai))

# Save the final globaldatatrainedon
np.save("processed_global/GLOBAL_Fused_Features.npy", X_fused_final)


print(f"AI Features Extracted & Fused")
print(f"Final Mega-Matrix Shape: {X_fused_final.shape}")


Difference data

Training Constrained Sparse Derivative AutoEncoder
Epoch [5/30] | Total Loss: 0.0319
Epoch [10/30] | Total Loss: 0.0315
Epoch [15/30] | Total Loss: 0.0311
Epoch [20/30] | Total Loss: 0.0283
Epoch [25/30] | Total Loss: 0.0257
Epoch [30/30] | Total Loss: 0.0241

Extracting AI Features and Fusing
AI Features Extracted & Fused
Final Mega-Matrix Shape: (200, 528)


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Global Training")

# Load the Master Data
X_mega = np.load("processed_global/GLOBAL_Fused_Features.npy")
y_mega = np.load("processed_global/GLOBAL_y.npy")

print(f"Loaded Mega-Matrix: {X_mega.shape}")

# 2. The 80/20 Global Split
# Stratify ensures we get an exactly even number of Left/Right/Feet/Tongue trials
X_train, X_test, y_train, y_test = train_test_split(
    X_mega, y_mega, test_size=0.2, random_state=42, stratify=y_mega
)

# The Forest Architecture
# Depth=10 allows it to navigate 192 features without memorizing noise
rf_global = RandomForestClassifier(
    n_estimators=1000, 
    max_depth=10,       
    max_features='sqrt', 
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

print("\nTraining 1000-Tree Random Forest...")
rf_global.fit(X_train, y_train)

# 4. The Final Test
y_pred = rf_global.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"FINAL GLOBAL ACCURACY: {accuracy * 100:.2f}%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
# Changed from 4 targets to 2!
target_names = ["Left Hand", "Right Hand"] 
print(classification_report(y_test, y_pred, target_names=target_names))

# --- THE FEATURE AUDIT ---
feature_names = []
phys_acoustic_labels = ['RMS', 'Var', 'Kurtosis', 'ZCR', 'Centroid', 'Rolloff', 'MFCC_1', 'MFCC_2']

# Update loop to 64 channels
for ch in range(64): 
    for label in phys_acoustic_labels:
        feature_names.append(f"Ch{ch+1}_{label}")

for i in range(16):
    feature_names.append(f"AI_Derivative_Latent_{i+1}")

importances = rf_global.feature_importances_
indices = np.argsort(importances)[-15:]

print("\n--- TOP 15 MOST INFLUENTIAL FEATURES ---")
for i in reversed(indices):
    print(f"{feature_names[i]:<25}: {importances[i]:.4f}")

Global Training
Loaded Mega-Matrix: (200, 528)

Training 1000-Tree Random Forest...
FINAL GLOBAL ACCURACY: 65.00%

Confusion Matrix:
[[13  7]
 [ 7 13]]

Classification Report:
              precision    recall  f1-score   support

   Left Hand       0.65      0.65      0.65        20
  Right Hand       0.65      0.65      0.65        20

    accuracy                           0.65        40
   macro avg       0.65      0.65      0.65        40
weighted avg       0.65      0.65      0.65        40


--- TOP 15 MOST INFLUENTIAL FEATURES ---
Ch49_ZCR                 : 0.0129
Ch32_MFCC_2              : 0.0108
Ch56_Var                 : 0.0100
Ch21_Var                 : 0.0096
Ch45_MFCC_2              : 0.0095
Ch43_MFCC_2              : 0.0089
AI_Derivative_Latent_4   : 0.0088
Ch11_ZCR                 : 0.0080
Ch31_MFCC_1              : 0.0079
Ch52_MFCC_2              : 0.0078
Ch21_ZCR                 : 0.0071
Ch46_ZCR                 : 0.0071
Ch44_Centroid            : 0.0068
Ch63_MFCC_2  

In [42]:
# import numpy as np
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler
# from sklearn.svm import SVC
# from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# print("Using SVM with RBF Kernel")

# # 1. Load the Mega-Matrix (Subject 1)
# X_mega = np.load("processed_global/GLOBAL_Fused_Features.npy")
# y_mega = np.load("processed_global/GLOBAL_y.npy")

# print(f"Loaded Mega-Matrix: {X_mega.shape}")

# # 2. Split the Data
# X_train, X_test, y_train, y_test = train_test_split(
#     X_mega, y_mega, test_size=0.2, random_state=42, stratify=y_mega
# )

# # 3. CRITICAL NEW STEP: Feature Standardization
# # This forces MFCCs, Variances, and AI Latents onto the exact same mathematical scale
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# # 4. The SVM (Support Vector Machine)
# # C=10 is a strict penalty for misclassification. 
# # kernel='rbf' allows it to draw curved boundaries instead of straight lines.
# svm_classifier = SVC(
#     kernel='rbf', 
#     C=10.0, 
#     gamma='scale', 
#     class_weight='balanced',
#     random_state=42
# )

# print("\nTraining RBF Support Vector Machine...")
# svm_classifier.fit(X_train_scaled, y_train)

# # 5. Evaluate
# y_pred = svm_classifier.predict(X_test_scaled)
# accuracy = accuracy_score(y_test, y_pred)

# print(f"FINAL SVM ACCURACY: {accuracy * 100:.2f}%")

# print("\nConfusion Matrix:")
# print(confusion_matrix(y_test, y_pred))

# print("\nClassification Report:")
# target_names = ["Left Hand", "Right Hand"]
# print(classification_report(y_test, y_pred, target_names=target_names))

In [69]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("RandomForestClassifer with Prebiased Features")

# 1. Load the Mega-Matrix
X_mega = np.load("processed_global/GLOBAL_Fused_Features.npy")
y_mega = np.load("processed_global/GLOBAL_y.npy")

# 2. Split the Data
X_train, X_test, y_train, y_test = train_test_split(
    X_mega, y_mega, test_size=0.2, random_state=42, stratify=y_mega
)

# 3. THE "PREBIAS" FILTER
# We statistically score all 528 features and ONLY keep the Top 40.
# The Random Forest will never even see the noisy/garbage features.
print("Applying Statistical Prebias (Keeping Top 40 Features)...")
selector = SelectKBest(score_func=f_classif, k=40)

# Learn the best 40 from the training data, and slice the matrices
X_train_biased = selector.fit_transform(X_train, y_train)
X_test_biased = selector.transform(X_test)

print(f"Old Shape: {X_train.shape[1]} features | New Biased Shape: {X_train_biased.shape[1]} features")

# 4. The Random Forest
rf_biased = RandomForestClassifier(
    n_estimators=1000, 
    max_depth=6,             
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

print("\nTraining Random Forest on Biased Features...")
rf_biased.fit(X_train_biased, y_train)

# 5. Evaluate
y_pred = rf_biased.predict(X_test_biased)
accuracy = accuracy_score(y_test, y_pred)

print(f"FINAL PREBIASED ACCURACY: {accuracy * 100:.2f}%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
target_names = ["Left Hand", "Right Hand"]
print(classification_report(y_test, y_pred, target_names=target_names))

RandomForestClassifer with Prebiased Features
Applying Statistical Prebias (Keeping Top 40 Features)...
Old Shape: 528 features | New Biased Shape: 40 features

Training Random Forest on Biased Features...
FINAL PREBIASED ACCURACY: 65.00%

Confusion Matrix:
[[12  8]
 [ 6 14]]

Classification Report:
              precision    recall  f1-score   support

   Left Hand       0.67      0.60      0.63        20
  Right Hand       0.64      0.70      0.67        20

    accuracy                           0.65        40
   macro avg       0.65      0.65      0.65        40
weighted avg       0.65      0.65      0.65        40



In [68]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("Cross validation with K folding")

# 1. Load the Mega-Matrix (Subject 1)
X_mega = np.load("processed_global/GLOBAL_Fused_Features.npy")
y_mega = np.load("processed_global/GLOBAL_y.npy")

print(f"Loaded Master Matrix: {X_mega.shape}")

# 2. The Optimized "Shallow" Forest
rf_optimized = RandomForestClassifier(
    n_estimators=1000, 
    max_depth=5,             
    max_features='log2',      
    min_samples_leaf=5,       
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

# 3. 5-Fold Cross Validation
# This ensures every single trial is used as a test case exactly once
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\nRunning 5-Fold Cross Validation...")
# This will train and test the model 5 separate times
scores = cross_val_score(rf_optimized, X_mega, y_mega, cv=cv, scoring='accuracy')

print("INDIVIDUAL FOLD SCORES:")
for i, score in enumerate(scores):
    print(f"   Fold {i+1}: {score * 100:.2f}%")

print("\nTRUE AVERAGE ACCURACY: {:.2f}% (+/- {:.2f}%)".format(
    scores.mean() * 100, scores.std() * 100))


Cross validation with K folding
Loaded Master Matrix: (200, 528)

Running 5-Fold Cross Validation...
INDIVIDUAL FOLD SCORES:
   Fold 1: 65.00%
   Fold 2: 62.50%
   Fold 3: 60.00%
   Fold 4: 52.50%
   Fold 5: 57.50%

TRUE AVERAGE ACCURACY: 59.50% (+/- 4.30%)


In [78]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

print("ONE LAST TRY!: XGBOOST GRID SEARCH")

# 1. Load the Master Data (The version where CAE had NO L1 penalty)
X_mega = np.load("processed_global/GLOBAL_Fused_Features.npy")
y_mega = np.load("processed_global/GLOBAL_y.npy")

# 2. The XGBoost Parameter Grid
param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    # THIS IS THE MAGIC: 
    # 0.1 forces it to use only ~52 features per tree, forcing it to find the Latents
    'colsample_bytree': [0.1, 0.3], 
    'subsample': [0.8] # Prevents overfitting to the "Dead Trials"
}

# 3. The Engine
xgb_base = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 4. The Grid Search
print("Testing 54 different XGBoost architectures")
grid_search = GridSearchCV(
    estimator=xgb_base, 
    param_grid=param_grid, 
    cv=cv, 
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_mega, y_mega)

print(f"BEST XGBOOST TRUE AVERAGE: {grid_search.best_score_ * 100:.2f}%")
print("\nOptimal XGBoost Settings:")
for param, value in grid_search.best_params_.items():
    print(f"   {param}: {value}")

# Let's peek at the feature importances of the winning model!
best_xgb = grid_search.best_estimator_
importances = best_xgb.feature_importances_
indices = np.argsort(importances)[-15:]

# Rebuild feature names to see if Latents survived
feature_names = []
phys_acoustic_labels = ['RMS', 'Var', 'Kurtosis', 'ZCR', 'Centroid', 'Rolloff', 'MFCC_1', 'MFCC_2']
for ch in range(64): 
    for label in phys_acoustic_labels:
        feature_names.append(f"Ch{ch+1}_{label}")
for i in range(8): # Remember we squeezed the bottleneck to 8!
    feature_names.append(f"AI_Derivative_Latent_{i+1}")

print("\n--- TOP 15 FEATURES (XGBOOST CHAMPION) ---")
for i in reversed(indices):
    # Just a safety check in case index is out of bounds
    if i < len(feature_names): 
        print(f"{feature_names[i]:<25}: {importances[i]:.4f}")

ONE LAST TRY!: XGBOOST GRID SEARCH
Testing 54 different XGBoost architectures... (Let him cook)
Fitting 5 folds for each of 54 candidates, totalling 270 fits
BEST XGBOOST TRUE AVERAGE: 64.50%

Optimal XGBoost Settings:
   colsample_bytree: 0.1
   learning_rate: 0.05
   max_depth: 5
   n_estimators: 100
   subsample: 0.8

--- TOP 15 FEATURES (XGBOOST CHAMPION) ---
Ch44_MFCC_1              : 0.0099
Ch45_Var                 : 0.0090
Ch37_Rolloff             : 0.0085
Ch41_Kurtosis            : 0.0080
Ch32_ZCR                 : 0.0079
Ch1_Rolloff              : 0.0078
Ch24_Rolloff             : 0.0078
Ch17_MFCC_1              : 0.0068
Ch38_Kurtosis            : 0.0067
Ch63_ZCR                 : 0.0064
Ch43_MFCC_2              : 0.0063
Ch36_Centroid            : 0.0062
Ch51_RMS                 : 0.0062
Ch44_Var                 : 0.0061
Ch36_MFCC_2              : 0.0061


In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from tqdm import tqdm # Gives us a nice progress bar

print("MONTE CARLO SIMULATION")

# 1. Load the Master Matrix
X_mega = np.load("processed_global/GLOBAL_Fused_Features.npy")
y_mega = np.load("processed_global/GLOBAL_y.npy")

print(f"Loaded Master Matrix: {X_mega.shape}")

iterations = 100
scores = []

print(f"\nRunning {iterations} Random Data Splits...")

# 2. The Loop
for i in tqdm(range(iterations)):
    # We change the random_state to 'i' so every loop is a new random shuffle
    X_train, X_test, y_train, y_test = train_test_split(
        X_mega, y_mega, test_size=0.2, random_state=i, stratify=y_mega
    )
    
    # Your best performing architecture
    rf = RandomForestClassifier(
        n_estimators=100,        # From Grid Search
        max_depth=7,             # From Grid Search (Shallow = safe from noise)
        max_features='log2',     # From Grid Search
        min_samples_leaf=5,      # From Grid Search
        class_weight='balanced',
        n_jobs=-1,
        random_state=42          # Keep the forest stable, data is shuffled by the loop
    )
    rf.fit(X_train, y_train)
    
    # Score and save
    acc = rf.score(X_test, y_test)
    scores.append(acc)

# 3. The Final Verdict
scores = np.array(scores)

print(f"100-RUN AVERAGE ACCURACY: {scores.mean() * 100:.2f}%")
print(f"PEAK ACCURACY ACHIEVED:   {scores.max() * 100:.2f}%")
print(f"LOWEST ACCURACY HIT:      {scores.min() * 100:.2f}%")

MONTE CARLO SIMULATION
Loaded Master Matrix: (200, 528)

Running 100 Random Data Splits...


100%|██████████| 100/100 [00:26<00:00,  3.71it/s]

100-RUN AVERAGE ACCURACY: 60.62%
PEAK ACCURACY ACHIEVED:   77.50%
LOWEST ACCURACY HIT:      40.00%


In [73]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

print("AUTOMATED ARCHITECTURE TUNING")

# 1. Load the Master Data
X_mega = np.load("processed_global/GLOBAL_Fused_Features.npy")
y_mega = np.load("processed_global/GLOBAL_y.npy")

# 2. The Parameter Grid (The gears we want to test)
param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [3, 5, 7, 10],
    'max_features': ['sqrt', 'log2'],
    'min_samples_leaf': [1, 3, 5]
}

# 3. The Base Model & Cross-Validator
rf_base = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 4. The Grid Search Engine
print("Testing 48 different Forest architectures... (This may take a minute)")
grid_search = GridSearchCV(
    estimator=rf_base, 
    param_grid=param_grid, 
    cv=cv, 
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

# Run the simulation!
grid_search.fit(X_mega, y_mega)

print(f"BEST TRUE AVERAGE ACCURACY: {grid_search.best_score_ * 100:.2f}%")
print("\nOptimal Architecture Settings:")
for param, value in grid_search.best_params_.items():
    print(f"   {param}: {value}")

AUTOMATED ARCHITECTURE TUNING
Testing 48 different Forest architectures... (This may take a minute)
Fitting 5 folds for each of 72 candidates, totalling 360 fits
BEST TRUE AVERAGE ACCURACY: 67.00%

Optimal Architecture Settings:
   max_depth: 7
   max_features: log2
   min_samples_leaf: 5
   n_estimators: 100
